# 22 · Color Clustering with K-Means

LRT4012 Robot Vision · Fall 2026


## Clusters group similar observations

$$J=\sum_{i=1}^{N}\|z_i-\mu_{c_i}\|_2^2.$$

$N$ is the number of pixels, $z_i$ is the color vector of pixel $i$, $c_i$ is its cluster index, $\mu_{c_i}$ is that cluster's center, and $J$ is the sum of squared distances minimized by k-means. $K$ specifies the number of clusters. Cluster IDs are arbitrary; one color cluster may contain several disconnected objects.

The example uses OpenCV's floating-point CIELAB conversion from BGR values in [0,1]. Channel scaling defines the distance being optimized. Compactness $J/N$ measures within-cluster variation, not semantic segmentation accuracy; increasing K usually decreases it.


In [ ]:
from pathlib import Path
import os, time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_repository():
    candidates = []
    if os.environ.get('VISION_ROBOTICA_REPO'):
        candidates.append(Path(os.environ['VISION_ROBOTICA_REPO']).expanduser())
    candidates += [Path.cwd(), *Path.cwd().parents]
    for root in candidates:
        if (root / 'requirements.txt').is_file() and (root / 'imagenes').is_dir():
            return root.resolve()
    raise FileNotFoundError('Open Jupyter inside the course repository, or set VISION_ROBOTICA_REPO to its root.')

REPO = find_repository()
IMAGE_DIR = REPO / 'imagenes'  # Read-only official resources.
# Save your own notebook and generated images in student_work/ and student_work/imagenes/.
cv2.setRNGSeed(4012)
cv2.setNumThreads(1)
rng = np.random.default_rng(4012)
plt.rcParams.update({'figure.dpi':100, 'font.size':10})

def load_image(name, max_side=640):
    image = cv2.imread(str(IMAGE_DIR / name), cv2.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(f'Official resource could not be read: {name}')
    h, w = image.shape[:2]
    if max(h, w) > max_side:
        scale = max_side / max(h, w)
        image = cv2.resize(image, (round(w*scale), round(h*scale)), interpolation=cv2.INTER_AREA)
    return image

def gray(image): return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

def panels(images, titles):
    fig, axes = plt.subplots(1, len(images), figsize=(4*len(images), 3.5), squeeze=False)
    for ax, im, title in zip(axes[0], images, titles):
        if im.ndim == 3:
            ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        else:
            ax.imshow(im, cmap='gray', vmin=0, vmax=255)
        ax.set_title(title)
        ax.axis('off')
    fig.tight_layout()
    plt.show()

def timed_ms(fn, repetitions=30):
    fn()  # Warm-up.
    times = []
    for _ in range(repetitions):
        start = time.perf_counter()
        fn()
        times.append((time.perf_counter()-start)*1000)
    return float(np.median(times))

def mask_metrics(prediction, reference):
    a, b = prediction.astype(bool), reference.astype(bool)
    tp = int(np.count_nonzero(a & b))
    fp = int(np.count_nonzero(a & ~b))
    fn = int(np.count_nonzero(~a & b))
    return {'IoU': tp/(tp+fp+fn) if tp+fp+fn else 1.0,
            'precision': tp/(tp+fp) if tp+fp else np.nan,
            'recall': tp/(tp+fn) if tp+fn else np.nan,
            'F1': 2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 1.0}


In [ ]:
image=load_image('pildoras2.png',max_side=360)
lab=cv2.cvtColor(image.astype(np.float32)/255,cv2.COLOR_BGR2LAB)
data=lab.reshape(-1,3)
rows=[]; views=[]
for K in [2,3,5]:
    cv2.setRNGSeed(4012)
    start=time.perf_counter()
    compactness,assignments,centers=cv2.kmeans(data,K,None,(cv2.TERM_CRITERIA_EPS+cv2.TERM_CRITERIA_MAX_ITER,50,0.2),5,cv2.KMEANS_PP_CENTERS)
    elapsed=(time.perf_counter()-start)*1000
    quantized=centers[assignments.ravel()].reshape(lab.shape)
    bgr=np.uint8(np.clip(cv2.cvtColor(quantized,cv2.COLOR_LAB2BGR)*255,0,255))
    rows.append({'K':K,'compactness_per_pixel':compactness/len(data),'single_run_ms':elapsed})
    views.append(bgr)
display(pd.DataFrame(rows).round(3))
panels([image,*views],['Input','K=2','K=3','K=5'])


## Robotic interpretation

Explain whether each cluster represents a color, a part, or background. State how connected components would be needed to count separated parts of the same color. Timing here is a labeled single-run observation, not the median benchmark.


## References

- [OpenCV: thresholding](https://docs.opencv.org/4.x/d7/d4d/tutorial_py_thresholding.html)
- [OpenCV: watershed](https://docs.opencv.org/4.13.0/d3/db4/tutorial_py_watershed.html)
- [OpenCV: contours](https://docs.opencv.org/4.x/d3/d05/tutorial_py_table_of_contents_contours.html)
- [OpenCV: k-means](https://docs.opencv.org/4.x/d1/d5c/tutorial_py_kmeans_opencv.html)
